# Day 030 — Exercise 5: Pipeline class

**What you'll build:** The `Pipeline` class — `add_step(name, fn)` registers steps, `run()` executes them with `chain_steps`, summarises with `summarize_run`, generates an AI report with `ai_pipeline_summary`, and returns a result dict with four keys: `name`, `steps`, `summary`, `report`.

**Why it matters:** The class encapsulates the step registry and orchestration logic behind a clean API. Adding a new automation step is one `add_step` call — no changes to the runner, no changes to the summary logic.

In [ ]:
import ollama
import time

## Provided: All Helper Functions

In [ ]:
import time


def run_step(name: str, fn) -> dict:
    start = time.time()
    try:
        result = fn()
        return {
            "name":       name,
            "status":     "ok",
            "result":     result,
            "error":      None,
            "duration_s": round(time.time() - start, 3),
        }
    except Exception as e:
        return {
            "name":       name,
            "status":     "error",
            "result":     None,
            "error":      str(e),
            "duration_s": round(time.time() - start, 3),
        }


def chain_steps(steps: list, stop_on_error: bool = True) -> list:
    results = []
    failed  = False
    for name, fn in steps:
        if failed and stop_on_error:
            results.append({
                "name":       name,
                "status":     "skipped",
                "result":     None,
                "error":      None,
                "duration_s": 0.0,
            })
        else:
            step_result = run_step(name, fn)
            results.append(step_result)
            if step_result["status"] == "error":
                failed = True
    return results


def summarize_run(step_results: list) -> dict:
    statuses = [s["status"] for s in step_results]
    return {
        "total":            len(step_results),
        "passed":           statuses.count("ok"),
        "failed":           statuses.count("error"),
        "skipped":          statuses.count("skipped"),
        "total_duration_s": round(
            sum(s.get("duration_s", 0.0) for s in step_results), 3
        ),
        "all_ok":           all(s == "ok" for s in statuses),
    }


def ai_pipeline_summary(step_results: list, model: str = "llama3.2") -> str:
    summary = summarize_run(step_results)
    lines = []
    for s in step_results:
        if s["status"] == "ok":
            lines.append(f"  \u2713 {s['name']} ({s['duration_s']:.3f}s)")
        elif s["status"] == "error":
            lines.append(f"  \u2717 {s['name']}: {s['error']}")
        else:
            lines.append(f"  - {s['name']}: skipped")
    run_text = (
        f"{summary['passed']}/{summary['total']} steps passed, "
        f"{summary['total_duration_s']}s total\n"
        + "\n".join(lines)
    )
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a pipeline monitor. "
                    "Summarise a workflow run in 2\u20133 sentences. Be concise."
                ),
            },
            {
                "role": "user",
                "content": f"{run_text}\n\nSummarise the run:",
            },
        ],
    )
    return response["message"]["content"]

## Your Implementation

In [ ]:
class Pipeline:
    def __init__(
        self,
        name: str = 'pipeline',
        stop_on_error: bool = True,
        model: str = 'llama3.2',
    ):
        # TODO: store name, stop_on_error, model
        # TODO: self._steps = []
        pass

    def add_step(self, name: str, fn) -> 'Pipeline':
        # TODO: append (name, fn) to self._steps
        # TODO: return self  (enables fluent chaining)
        pass

    def run(self) -> dict:
        # TODO: step_results = chain_steps(self._steps, stop_on_error=self.stop_on_error)
        # TODO: summary = summarize_run(step_results)
        # TODO: report  = ai_pipeline_summary(step_results, model=self.model)
        # TODO: return {'name': self.name, 'steps': step_results,
        #               'summary': summary, 'report': report}
        pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: class and methods defined
    try:
        assert 'Pipeline' in globals()
        for m in ('add_step', 'run'):
            assert hasattr(Pipeline, m), f'Pipeline missing method: {m}'
        passed += 1; print('\u2705 Check 1: Pipeline class with add_step and run')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}')
        return

    p = None

    # Check 2: add_step returns self (fluent chaining)
    try:
        p = Pipeline(name='test_pipe')
        ret = p.add_step('step_a', lambda: 1)
        assert ret is p, \
            f'add_step should return self, got {type(ret)}'
        passed += 1; print('\u2705 Check 2: add_step returns self')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: _steps contains the registered steps
    try:
        assert p is not None
        p.add_step('step_b', lambda: 2)
        assert hasattr(p, '_steps'), 'Pipeline missing _steps attribute'
        assert len(p._steps) == 2, \
            f'expected 2 steps, got {len(p._steps)}'
        passed += 1; print('\u2705 Check 3: _steps has 2 entries after 2 add_step calls')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: run() returns dict with all four keys
    try:
        result = p.run()
        assert isinstance(result, dict), \
            f'run() should return dict, got {type(result)}'
        for k in ('name', 'steps', 'summary', 'report'):
            assert k in result, f"result missing key: '{k}'"
        passed += 1; print('\u2705 Check 4: run() returns dict with name/steps/summary/report')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: summary.all_ok=True and name is preserved
    try:
        result = Pipeline(name='my_pipe').add_step('only', lambda: 42).run()
        assert result['name'] == 'my_pipe', \
            f"name wrong: {result['name']!r}"
        assert result['summary']['all_ok'] is True, \
            f"all_ok wrong: {result['summary']['all_ok']}"
        assert len(result['steps']) == 1
        assert isinstance(result['report'], str) and len(result['report']) > 5
        passed += 1; print('\u2705 Check 5: summary.all_ok=True; name and report correct')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
class Pipeline:
    def __init__(
        self,
        name: str = "pipeline",
        stop_on_error: bool = True,
        model: str = "llama3.2",
    ):
        self.name          = name
        self.stop_on_error = stop_on_error
        self.model         = model
        self._steps: list  = []

    def add_step(self, name: str, fn) -> "Pipeline":
        self._steps.append((name, fn))
        return self

    def run(self) -> dict:
        step_results = chain_steps(self._steps, stop_on_error=self.stop_on_error)
        summary      = summarize_run(step_results)
        report       = ai_pipeline_summary(step_results, model=self.model)
        return {
            "name":    self.name,
            "steps":   step_results,
            "summary": summary,
            "report":  report,
        }
```

</details>